# FASE 6 - Multimodal: Imagem

Status atualizado em `2026-03-26`.

Este documento deixou de ser apenas proposta inicial e passa a registrar o baseline implementado, as decisoes aprovadas e o backlog restante da fase 6.


## 1. Objetivo da fase

Entregar um modulo local de geracao de imagens capaz de:

- gerar imagem por texto;
- gerar imagem por referencia;
- manter continuidade visual por projeto;
- atender casos como ilustracao de historias, arte para musica e conjuntos visuais.

A expressao `sem limites de quantidade` continua significando sem limite artificial por API. Na pratica, os limites reais sao GPU, RAM, tempo de inferencia e armazenamento.


## 2. Decisoes aprovadas

Direcao tecnica consolidada:

- `diffusers` como engine principal;
- `ComfyUI` tratado como interface externa opcional do MVP, nao como dependencia obrigatoria do runtime interno;
- pesos fora do Git do repositorio;
- baseline local com `SD 1.5-class`;
- `img2img` e `inpainting` como fluxos centrais do MVP;
- `IP-Adapter` e `1 ControlNet por vez` como extensoes controladas;
- workflow operacional: `rascunho pequeno -> selecao -> refinamento -> upscale`;
- `SDXL` reservado para imagem final selecionada, nao para fluxo diario;
- `FLUX`, LoRA pesada e fine-tuning fora do MVP.


## 3. Perfil de hardware alvo

O baseline foi alinhado para a maquina atual:

- `Intel i7-10750H`
- `16 GB RAM`
- `RTX 2060 6 GB`

Defaults operacionais assumidos hoje:

- `512x512`
- `18 steps` no rascunho
- `guidance scale = 7.5` no rascunho
- `2 imagens por vez`

No fluxo de qualidade atual, `img2img` usa `20 steps`, `guidance 7.2`, `strength 0.30` e `1 imagem`, enquanto `inpainting` usa `24 steps`, `guidance 7.5`, `strength 0.75`, `mask blur 6.0` e `1 imagem`.


## 4. Arquitetura implementada

### 4.1. Organizacao de codigo

- `tools/ml/`: pipeline de ML
- `tools/img_gen/`: pipeline de imagem
- `tools/common/`: utilitarios compartilhaveis, quando necessario

### 4.2. Persistencia

- `projects/img_gen/<project_id>/...`: estado duravel do projeto
- `artifacts/img_gen/<project_id>/run_<timestamp>/...`: artefatos de execucao por projeto
- `artifacts/img_gen/_ad_hoc/run_<timestamp>/...`: geracao livre sem projeto
- `models/img_gen/...`: pesos e registry local

### 4.3. Entidade central

A fase 6 passou a ser centrada em `project_id`, nao apenas em `run_id`.


## 5. Modelo de projeto

Cada projeto pode concentrar:

- `project.json`
- `visual_bibles/manifest.json`
- `references/manifest.json`
- `imported_projects/manifest.json`
- `promoted_assets/manifest.json`
- `derived_context/manifest.json`
- `working_sets/manifest.json`

Principios aprovados:

- `visual_bible` versionado;
- projetos importados por manifesto-ponta, com selecao parcial de assets;
- snapshot local somente quando o asset for promovido ou congelado no projeto atual;
- imagens geradas anteriormente nao entram automaticamente como referencia: o usuario decide o que validar e promover.


## 6. Estado implementado do runtime de imagem

Hoje o runtime interno ja suporta:

- `txt2img`
- `img2img`
- `inpainting`
- manifests por run
- multiplas imagens por execucao
- `safety_checker` ligavel/desligavel por execucao
- scheduler de melhor qualidade para SD 1.5 (`DPM++ 2M Karras`)
- quality preset automatico com reforco de prompt e negative prompt tecnico
- blur automatico de mascara no `inpainting`
- registro de modelos locais por alias
- checkpoint base `sd15_runwayml_base`
- modelo dedicado `sd15_inpainting_base`

Tambem ja existe suporte estrutural para:

- `IP-Adapter`
- `ControlNet`

Mas esses modulos ainda dependem do download e registro dos pesos locais correspondentes.


## 7. Estado implementado da webapp

A interface web atual ja foi simplificada para uso leigo.

Pontos implementados:

- aba `Geracao` como entrada padrao;
- geracao livre sem projeto;
- `Rascunho`, `Refino`, `Inpainting` e `Resultados` em fluxo simples;
- projeto opcional, com aba `Projeto` separada;
- `Contexto` simplificado em `Visual Bible`, `Referencias`, `Imports` e `Working Set e Biblioteca`;
- busca simples e acoes rapidas na biblioteca;
- upload direto de referencias, imagem-base e mascara;
- importacao visual de imagens de outro projeto;
- promocao simples de resultados aprovados para o projeto.


## 8. Reforco de consistencia visual ja implementado

O sistema passou a compor prompts com contexto do `visual_bible` ativo.

Hoje entram no prompt composto:

- resumo do `visual_bible`;
- `prompt_anchor` e `negative_prompt_anchor`;
- `style_profile`;
- personagens relevantes ao prompt;
- ambientes relevantes ao prompt;
- regras de composicao;
- regras de continuidade.

Isso melhorou aderencia, mas ainda nao equivale a consistencia perfeita de personagem. Para esse nivel, ainda podem ser necessarios referencia visual mais forte, `IP-Adapter` ou pesos adicionais em fases futuras.


## 9. Modelos, pesos e armazenamento

Layout local aprovado:

```text
models/
  img_gen/
    checkpoints/
    inpainting/
    controlnet/
    ip_adapter/
    sam/
    upscalers/
    cache/
```

Arquivos fora do Git:

- `models/`
- `external/ComfyUI/` quando usado

Registry local:

- `models/img_gen/registry.json`


## 10. Entregas ja concluidas na fase 6

### Infraestrutura

- reorganizacao de `tools/` em `tools/ml` e `tools/img_gen`;
- separacao clara entre `projects/img_gen` e `artifacts/img_gen`;
- registry de modelos locais;
- runtime e validacao de ambiente para `diffusers`.

### Projeto e contexto

- nucleo de `project_id`;
- `visual_bible` versionado;
- referencias com metadata;
- imports por projeto de origem;
- `working_set` e `promoted_assets`.

### Geracao

- `txt2img` real funcionando localmente;
- `img2img` real funcionando localmente;
- `inpainting` funcional quando ha checkpoint dedicado.

### UX

- webapp simplificada para fluxo leigo;
- projeto de onboarding pronto;
- manual de uso atualizado.


## 11. Itens pendentes prioritarios

Backlog mais importante a partir do estado atual:

1. testar e registrar um primeiro `IP-Adapter` end-to-end;
2. testar e registrar um primeiro `ControlNet` end-to-end;
3. melhorar ainda mais a aderencia do `visual_bible` no `Refino` e no `Inpainting`;
4. avaliar geracao de mascara assistida por `MobileSAM`;
5. decidir como expor `SDXL` para imagem final sem poluir o fluxo simples;
6. unificar futuramente as webapps de ML e imagem em um `main_app.py` ou estrutura equivalente.


## 12. Riscos e limites atuais

- `SD 1.5-class` ainda pode falhar em obediencia fina de personagem sem referencia visual forte;
- qualidade e consistencia ainda dependem de iteracao humana no fluxo `rascunho -> selecao -> refinamento`;
- `IP-Adapter` e `ControlNet` exigem pesos locais adicionais;
- o perfil `RTX 2060 6 GB` continua exigindo cuidado com VRAM e combinacao de modulos;
- a fase 6 ainda nao entrou em fine-tuning, LoRA por projeto nem identidade facial dedicada.


## 13. Criterio de aceite revisado

A fase 6 ja atingiu o baseline minimo quando:

- ha geracao local funcionando sem API externa;
- existe fluxo web simples para gerar, refinar e editar por mascara;
- existe `project_id` com persistencia de contexto;
- prompts, seeds, modelos e imagens ficam registrados por run;
- o usuario consegue gerar imagem sem preencher parametros tecnicos;
- o projeto consegue crescer depois com `IP-Adapter`, `ControlNet` e modulos extras sem quebrar a base atual.

Pelo estado atual do repositorio, esse baseline ja esta implementado. O restante da fase 6 agora e aprofundamento de consistencia, referencia e ergonomia.
